# Taller: De Pixels a Coordenadas
## Explorando la Imagen como Matriz

**Herramientas:** `opencv-python`, `numpy`, `matplotlib`

---
## 0. Instalación e importaciones

In [ ]:
# Instalar dependencias (solo necesario en Colab)
# !pip install opencv-python numpy matplotlib

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Configuración global de plots
plt.rcParams['figure.facecolor'] = '#0f0f0f'
plt.rcParams['axes.facecolor'] = '#1a1a2e'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = 'white'
plt.rcParams['xtick.color'] = 'white'
plt.rcParams['ytick.color'] = 'white'
plt.rcParams['axes.titlecolor'] = 'white'
plt.rcParams['axes.edgecolor'] = '#444'

def mostrar(titulo, img_bgr=None, img_rgb=None, cmap=None, figsize=(6,5)):
    """Helper para mostrar imágenes con matplotlib."""
    fig, ax = plt.subplots(figsize=figsize)
    if img_bgr is not None:
        ax.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    elif img_rgb is not None:
        ax.imshow(img_rgb, cmap=cmap)
    ax.set_title(titulo, fontsize=13, pad=10)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

print('✅ Librerías cargadas correctamente')
print(f'   OpenCV versión: {cv2.__version__}')
print(f'   NumPy versión:  {np.__version__}')

---
## 1. Cargar la imagen y explorar su estructura matricial

Usamos `cv2.imread()` que carga la imagen en formato **BGR** (Blue-Green-Red),  
a diferencia de matplotlib que espera **RGB**. ¡Cuidado con este detalle!

In [ ]:
# ── Cargar imagen ──────────────────────────────────────────────────
ruta = 'media/imagen_taller.png'   # ← cambia aquí si usas otra imagen
img_bgr = cv2.imread(ruta)

if img_bgr is None:
    raise FileNotFoundError(f'No se encontró la imagen en: {ruta}')

# ── Información de la matriz ───────────────────────────────────────
alto, ancho, canales = img_bgr.shape
print('═' * 45)
print('  INFORMACIÓN DE LA IMAGEN')
print('═' * 45)
print(f'  Tipo de dato  : {img_bgr.dtype}')
print(f'  Forma (shape) : {img_bgr.shape}  →  (alto, ancho, canales)')
print(f'  Alto          : {alto} px')
print(f'  Ancho         : {ancho} px')
print(f'  Canales       : {canales}  (B, G, R en OpenCV)')
print(f'  Total píxeles : {alto * ancho:,}')
print(f'  Memoria aprox : {img_bgr.nbytes / 1024:.1f} KB')
print('═' * 45)

# ── Inspección de píxeles concretos ───────────────────────────────
print('\n  VALORES DE PÍXELES INDIVIDUALES')
print(f'  Píxel [50, 50]    (cielo)  : BGR = {img_bgr[50,  50]}')
print(f'  Píxel [300, 250]  (casa)   : BGR = {img_bgr[300, 250]}')
print(f'  Píxel [300, 430]  (árbol)  : BGR = {img_bgr[300, 430]}')

mostrar('Imagen original (BGR → RGB para display)', img_bgr=img_bgr, figsize=(6,6))

---
## 2. Canales de color: RGB y HSV

### 2.1 Separar canales RGB
Cada canal es una **matriz 2D** de intensidades. Podemos separarlos con `cv2.split()`  
o con slicing de NumPy: `img[:, :, 0]` → canal azul.

In [ ]:
# Convertir BGR → RGB para trabajar intuitivamente
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

# Separar los tres canales
canal_r = img_rgb[:, :, 0]   # Red
canal_g = img_rgb[:, :, 1]   # Green
canal_b = img_rgb[:, :, 2]   # Blue

# También podemos usar cv2.split() que devuelve (B, G, R) en OpenCV
b, g, r = cv2.split(img_bgr)

# Visualizar canales en sus colores reales
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

datos = [
    (img_rgb,  'Imagen RGB\nOriginal',  None),
    (canal_r,  'Canal R (Rojo)',         'Reds'),
    (canal_g,  'Canal G (Verde)',        'Greens'),
    (canal_b,  'Canal B (Azul)',         'Blues'),
]

for ax, (data, titulo, cmap) in zip(axes, datos):
    ax.imshow(data, cmap=cmap)
    ax.set_title(titulo, fontsize=11)
    ax.axis('off')
    # Anotación con estadísticas
    if cmap is not None:
        ax.set_xlabel(f'min={data.min()}  max={data.max()}  media={data.mean():.1f}',
                      color='#aaa', fontsize=9)

fig.suptitle('Descomposición en Canales RGB', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print('Dimensión de cada canal:', canal_r.shape, '← solo (alto, ancho), sin canal')
print('Ejemplo canal_r[50, 50] =', canal_r[50, 50], '→ intensidad del rojo en ese píxel')

### 2.2 Espacio de color HSV

HSV descompone el color de forma más intuitiva para humanos:
- **H** (Hue / Matiz): tipo de color, ángulo 0-179 en OpenCV (0=rojo, 60=verde, 120=azul)
- **S** (Saturation / Saturación): pureza del color, 0-255
- **V** (Value / Brillo): luminosidad, 0-255

Es muy útil para **segmentar colores** (ej. detectar objetos por su color).

In [ ]:
# Convertir BGR → HSV
img_hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)

canal_h, canal_s, canal_v = cv2.split(img_hsv)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

datos_hsv = [
    (img_rgb,   'Imagen RGB\nOriginal',  None),
    (canal_h,   'Canal H\n(Matiz)',       'hsv'),
    (canal_s,   'Canal S\n(Saturación)', 'plasma'),
    (canal_v,   'Canal V\n(Valor/Brillo)', 'gray'),
]

for ax, (data, titulo, cmap) in zip(axes, datos_hsv):
    ax.imshow(data, cmap=cmap)
    ax.set_title(titulo, fontsize=11)
    ax.axis('off')

fig.suptitle('Descomposición en Canales HSV', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Tabla comparativa de píxeles en ambos espacios de color
print('\n  COMPARACIÓN DE ESPACIOS DE COLOR EN PUNTOS CLAVE')
print(f'  {"Región":<20} {"BGR":>20} {"HSV":>20}')
print('  ' + '─' * 60)
puntos = [
    ('Cielo (50,50)',    (50, 50)),
    ('Sol (80, 400)',    (80, 400)),
    ('Casa (300,250)',   (300, 250)),
    ('Árbol (290,432)', (290, 432)),
]
for nombre, (y, x) in puntos:
    bgr_val = img_bgr[y, x]
    hsv_val = img_hsv[y, x]
    print(f'  {nombre:<20} BGR={str(bgr_val):>17}   HSV={str(hsv_val):>17}')

---
## 3. Slicing de matrices: Modificar regiones específicas

En NumPy, el slicing funciona igual que en listas pero en múltiples dimensiones:
```python
imagen[y_inicio:y_fin, x_inicio:x_fin]  →  submatriz (región rectangular)
imagen[y_inicio:y_fin, x_inicio:x_fin] = [B, G, R]  →  asignar color
```

### 3.1 Cambiar el color de un área rectangular

In [ ]:
# Trabajamos sobre una copia para no modificar el original
img_mod1 = img_bgr.copy()

# ─── Modificación 1: Ventana izquierda → rojo brillante ──────────
# Coordenadas: [fila_inicio:fila_fin, col_inicio:col_fin]
img_mod1[275:320, 165:215] = [0, 0, 255]   # BGR rojo

# ─── Modificación 2: Ventana derecha → cyan ───────────────────────
img_mod1[275:320, 290:345] = [255, 255, 0]  # BGR cyan

# ─── Modificación 3: Zona del suelo → naranja ─────────────────────
img_mod1[450:510, 50:200] = [0, 140, 255]   # BGR naranja

# ─── Dibujar rectángulos para señalar las zonas ───────────────────
img_anotada = img_mod1.copy()
cv2.rectangle(img_anotada, (165, 275), (215, 320), (255, 255, 255), 2)
cv2.rectangle(img_anotada, (290, 275), (345, 320), (255, 255, 255), 2)
cv2.rectangle(img_anotada, (50, 450),  (200, 510), (255, 255, 255), 2)

# Etiquetas
cv2.putText(img_anotada, 'Rojo', (167, 310), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255,255,255), 1)
cv2.putText(img_anotada, 'Cyan', (292, 310), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255,255,255), 1)
cv2.putText(img_anotada, 'Naranja', (55, 490), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 1)

# Comparación lado a lado
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 6))
ax1.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
ax1.set_title('Original', fontsize=12)
ax1.axis('off')
ax2.imshow(cv2.cvtColor(img_anotada, cv2.COLOR_BGR2RGB))
ax2.set_title('Después del slicing\n(regiones recoloreadas)', fontsize=12)
ax2.axis('off')
plt.suptitle('Modificación de Regiones con Slicing NumPy', fontsize=14)
plt.tight_layout()
plt.show()

print('Código equivalente al slicing de ventana izquierda:')
print('  img_mod1[275:320, 165:215] = [0, 0, 255]')
print('  ↑fila_ini ↑fila_fin ↑col_ini ↑col_fin   ↑valor BGR')
print(f'  Tamaño del bloque: {(320-275)} filas × {(215-165)} columnas = {(320-275)*(215-165)} píxeles modificados')

### 3.2 Sustituir una región por otra parte de la imagen

El slicing también permite **copiar bloques completos** de una región a otra.  
Los bloques deben tener el **mismo tamaño** (mismo número de filas y columnas).

In [ ]:
img_mod2 = img_bgr.copy()

# ─── Recortar región fuente: el sol ──────────────────────────────
# Sol centrado aprox en (80, 400), radio 60 → bounding box 120×120
y1_src, y2_src = 20,  140
x1_src, x2_src = 340, 460
region_sol = img_bgr[y1_src:y2_src, x1_src:x2_src].copy()

# ─── Pegar el sol en dos nuevas posiciones ───────────────────────
h, w = region_sol.shape[:2]
print(f'Tamaño de la región copiada (sol): {h}×{w} px  →  shape {region_sol.shape}')

# Posición 1: esquina superior izquierda
img_mod2[20:20+h, 20:20+w] = region_sol

# Posición 2: en el cielo central
img_mod2[30:30+h, 196:196+w] = region_sol

# ─── También copiamos un árbol clonado ───────────────────────────
arbol_src = img_bgr[230:390, 395:490].copy()
h2, w2 = arbol_src.shape[:2]
img_mod2[230:230+h2, 20:20+w2] = arbol_src

# Anotaciones
img_anot2 = img_mod2.copy()
cv2.rectangle(img_anot2, (x1_src, y1_src), (x2_src, y2_src), (0, 255, 255), 2)
cv2.putText(img_anot2, 'ORIGEN', (x1_src, y1_src-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,255), 1)
cv2.rectangle(img_anot2, (20, 20), (20+w, 20+h), (0, 255, 0), 2)
cv2.putText(img_anot2, 'COPIA 1', (22, 18), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 1)
cv2.rectangle(img_anot2, (196, 30), (196+w, 30+h), (0, 255, 0), 2)
cv2.putText(img_anot2, 'COPIA 2', (198, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 6))
ax1.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
ax1.set_title('Original', fontsize=12)
ax1.axis('off')
ax2.imshow(cv2.cvtColor(img_anot2, cv2.COLOR_BGR2RGB))
ax2.set_title('Sustitución de regiones\n(cian=origen, verde=destinos)', fontsize=12)
ax2.axis('off')
plt.suptitle('Copy-Paste de Regiones con Slicing', fontsize=14)
plt.tight_layout()
plt.show()

print('\nCódigo del copy-paste:')
print('  region = img[y1:y2, x1:x2].copy()   # extraer')
print('  img[y1_dst:y2_dst, x1_dst:x2_dst] = region  # pegar')

---
## 4. Histograma de Intensidades

El histograma muestra la **distribución de valores de brillo/color** en la imagen.  
- Eje X: valor de intensidad (0=negro … 255=blanco)
- Eje Y: número de píxeles con ese valor

Un histograma concentrado a la **izquierda** → imagen oscura.  
Concentrado a la **derecha** → imagen sobreexpuesta.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

# ─── Histograma con cv2.calcHist() ────────────────────────────────
ax = axes[0, 0]
colores = ('b', 'g', 'r')
nombres = ('Azul', 'Verde', 'Rojo')
for canal_idx, (color, nombre) in enumerate(zip(colores, nombres)):
    # cv2.calcHist(images, channels, mask, histSize, ranges)
    hist = cv2.calcHist([img_bgr], [canal_idx], None, [256], [0, 256])
    ax.plot(hist, color=color, alpha=0.8, linewidth=1.5, label=nombre)
ax.set_title('Histograma RGB\n(cv2.calcHist)', fontsize=11)
ax.set_xlabel('Intensidad (0-255)')
ax.set_ylabel('Número de píxeles')
ax.legend()
ax.set_xlim([0, 256])

# ─── Histograma con matplotlib.hist() ─────────────────────────────
ax2 = axes[0, 1]
for canal_idx, (color, nombre) in enumerate(zip(colores, nombres)):
    canal_data = img_bgr[:, :, canal_idx].ravel()  # aplanar a 1D
    ax2.hist(canal_data, bins=256, range=(0, 256), color=color,
             alpha=0.5, label=nombre, histtype='stepfilled')
ax2.set_title('Histograma RGB\n(matplotlib.hist)', fontsize=11)
ax2.set_xlabel('Intensidad (0-255)')
ax2.set_ylabel('Frecuencia')
ax2.legend()

# ─── Histograma en escala de grises ───────────────────────────────
ax3 = axes[1, 0]
img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
hist_gray = cv2.calcHist([img_gray], [0], None, [256], [0, 256])
ax3.fill_between(range(256), hist_gray.flatten(), color='#bb86fc', alpha=0.7)
ax3.plot(hist_gray, color='#e040fb', linewidth=1.5)
ax3.set_title('Histograma Escala de Grises', fontsize=11)
ax3.set_xlabel('Intensidad (0-255)')
ax3.set_ylabel('Número de píxeles')
ax3.set_xlim([0, 256])

# ─── Imagen en grises como referencia ─────────────────────────────
ax4 = axes[1, 1]
ax4.imshow(img_gray, cmap='gray')
ax4.set_title('Imagen en Escala de Grises', fontsize=11)
ax4.axis('off')

plt.suptitle('Histogramas de Intensidad', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

# Estadísticas
print('ESTADÍSTICAS DE LA IMAGEN (canal a canal BGR):')
for i, nombre in enumerate(['Azul', 'Verde', 'Rojo']):
    c = img_bgr[:,:,i]
    print(f'  {nombre:6}: min={c.min():3d}  max={c.max():3d}  '
          f'media={c.mean():5.1f}  std={c.std():5.1f}')

---
## 5. Ajuste de Brillo y Contraste

La ecuación fundamental del ajuste lineal es:

$$g(x,y) = \alpha \cdot f(x,y) + \beta$$

- **α (alpha)**: controla el **contraste** (ganancia). α > 1 → más contraste; α < 1 → menos contraste
- **β (beta)**: controla el **brillo** (sesgo). β > 0 → más brillo; β < 0 → más oscuro

Los valores se **saturan** automáticamente al rango [0, 255].

In [ ]:
# ─── Función manual con NumPy (usando clip para saturar) ──────────
def ajuste_manual(imagen, alpha, beta):
    """g = alpha * f + beta  (saturado a [0, 255])"""
    resultado = np.clip(alpha * imagen.astype(np.float32) + beta, 0, 255)
    return resultado.astype(np.uint8)

# ─── Función de OpenCV (más eficiente) ────────────────────────────
# cv2.convertScaleAbs(src, alpha=1, beta=0)

configuraciones = [
    ('Original',              img_bgr,                                      1.0,  0),
    ('Más brillo (+80)',      cv2.convertScaleAbs(img_bgr, alpha=1.0, beta=80),  1.0, 80),
    ('Menos brillo (-80)',    cv2.convertScaleAbs(img_bgr, alpha=1.0, beta=-80), 1.0,-80),
    ('Más contraste (α=1.8)', cv2.convertScaleAbs(img_bgr, alpha=1.8, beta=0),  1.8,  0),
    ('Menos contraste (α=0.5)', cv2.convertScaleAbs(img_bgr, alpha=0.5, beta=0), 0.5, 0),
    ('Manual: α=1.5, β=30',   ajuste_manual(img_bgr, 1.5, 30),               1.5, 30),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, (titulo, img_var, alpha, beta) in zip(axes.flat, configuraciones):
    ax.imshow(cv2.cvtColor(img_var, cv2.COLOR_BGR2RGB))
    ax.set_title(f'{titulo}', fontsize=10, pad=6)
    ax.axis('off')
    # Calcular estadísticas de brillo
    gray = cv2.cvtColor(img_var, cv2.COLOR_BGR2GRAY)
    ax.set_xlabel(f'Brillo medio: {gray.mean():.1f}  |  Std: {gray.std():.1f}',
                  color='#aaa', fontsize=8)

plt.suptitle('Ajustes de Brillo (β) y Contraste (α)\nFórmula: g(x,y) = α·f(x,y) + β',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ─── Comparar histogramas antes/después del ajuste ────────────────
original   = img_bgr
mas_brillo = cv2.convertScaleAbs(img_bgr, alpha=1.0, beta=80)
mas_contr  = cv2.convertScaleAbs(img_bgr, alpha=1.8, beta=0)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

pares = [
    (original,   'Original'),
    (mas_brillo, 'Más brillo (β=+80)'),
    (mas_contr,  'Más contraste (α=1.8)'),
]

for col, (img_var, titulo) in enumerate(pares):
    # Imagen
    axes[0, col].imshow(cv2.cvtColor(img_var, cv2.COLOR_BGR2RGB))
    axes[0, col].set_title(titulo, fontsize=11)
    axes[0, col].axis('off')
    # Histograma
    for i, color in enumerate(['b','g','r']):
        hist = cv2.calcHist([img_var], [i], None, [256], [0,256])
        axes[1, col].plot(hist, color=color, alpha=0.75, linewidth=1.2)
    axes[1, col].set_xlim([0, 256])
    axes[1, col].set_xlabel('Intensidad')
    if col == 0:
        axes[1, col].set_ylabel('Píxeles')
    # Línea vertical en la media de grises
    gv = cv2.cvtColor(img_var, cv2.COLOR_BGR2GRAY).mean()
    axes[1, col].axvline(gv, color='yellow', linestyle='--', linewidth=1.2, label=f'media={gv:.0f}')
    axes[1, col].legend(fontsize=8)

plt.suptitle('Efecto de los Ajustes sobre el Histograma', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print('Observa cómo:')
print('  • +Brillo  → histograma se desplaza a la DERECHA')
print('  • +Contraste → histograma se ENSANCHA (mayor dispersión)')